In [1]:
import os
import zipfile
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [5]:
extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [6]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/mmWave_data
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/pitch
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/height


In [8]:
author_dir = "/content/Image beam"
os.makedirs(author_dir, exist_ok=True)


print(author_dir)

/content/Image beam


In [9]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"

In [10]:
DATA_ROOT = "/content/dataset/scenario23_dev"   # update if needed
print("Exists:", os.path.exists(DATA_ROOT))

Exists: True


In [11]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"  # adjust path

needed = [
    "build_net.py",
    "data_feed.py",
    "main_beam.py",
    "main_beam_eval.py",
    "scenario23_img_beam_train.csv",
    "scenario23_img_beam_val.csv",
    "scenario23_img_beam_test.csv",
]

for f in needed:
    p = os.path.join(AUTHOR_ROOT, f)
    print(f, "->", os.path.exists(p), p)

build_net.py -> True /content/drive/MyDrive/Image beam/build_net.py
data_feed.py -> True /content/drive/MyDrive/Image beam/data_feed.py
main_beam.py -> True /content/drive/MyDrive/Image beam/main_beam.py
main_beam_eval.py -> True /content/drive/MyDrive/Image beam/main_beam_eval.py
scenario23_img_beam_train.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_train.csv
scenario23_img_beam_val.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_val.csv
scenario23_img_beam_test.csv -> True /content/drive/MyDrive/Image beam/scenario23_img_beam_test.csv


In [12]:
train_csv = os.path.join(AUTHOR_ROOT, "scenario23_img_beam_train.csv")
df_img_author = pd.read_csv(train_csv)

print(df_img_author.head())
print(df_img_author.columns.tolist())

   index                                          unit1_rgb  unit1_beam
0   3532  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
1   2224  ../scenario23_dev/unit1/camera_data/image_BS1_...          14
2   9416  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
3   8510  ../scenario23_dev/unit1/camera_data/image_BS1_...          20
4   6877  ../scenario23_dev/unit1/camera_data/image_BS1_...          17
['index', 'unit1_rgb', 'unit1_beam']
